# Bench Press vs Shoulder Press — IMU signal comparison

Compares wrist-worn IMU signal patterns between two RecoFit exercises:
**Chest Press (rack)** ("bench press") and **Squat Rack Shoulder Press**.

**Data required:** `data/recordings.csv` and `data/samples.csv` (see
`data/README.md` for what they are and how they were produced). This
notebook has not been run against real data yet — fill `data/` first,
then run all cells top to bottom.

## 1. Load data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

DATA_DIR = "../data"

recordings = pd.read_csv(f"{DATA_DIR}/recordings.csv")
samples = pd.read_csv(f"{DATA_DIR}/samples.csv")

print(recordings.shape, samples.shape)
recordings.head()

## 2. Select the two exercises to compare

In [ ]:
BENCH = "Chest Press (rack)"
SHOULDER = "Squat Rack Shoulder Press"

recordings["activity_name"].value_counts()

In [ ]:
bench_recordings = recordings[recordings["activity_name"] == BENCH]
shoulder_recordings = recordings[recordings["activity_name"] == SHOULDER]

print(f"{BENCH}: {len(bench_recordings)} recordings")
print(f"{SHOULDER}: {len(shoulder_recordings)} recordings")

## 3. Pick one example recording per exercise and plot the raw signal

Accelerometer magnitude (`sqrt(x^2 + y^2 + z^2)`) over time, one clean
example set per exercise.

In [ ]:
def accel_magnitude(record_uid: str) -> pd.DataFrame:
    rec = samples[samples["record_uid"] == record_uid].sort_values("sample_index")
    mag = np.sqrt(rec["accel_x_g"]**2 + rec["accel_y_g"]**2 + rec["accel_z_g"]**2)
    return pd.DataFrame({"time_s": rec["time_s"].to_numpy(), "accel_mag_g": mag.to_numpy()})

bench_uid = bench_recordings.iloc[0]["record_uid"]
shoulder_uid = shoulder_recordings.iloc[0]["record_uid"]

bench_sig = accel_magnitude(bench_uid)
shoulder_sig = accel_magnitude(shoulder_uid)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=False)

axes[0].plot(bench_sig["time_s"], bench_sig["accel_mag_g"])
axes[0].set_title(f"{BENCH} — {bench_uid}")
axes[0].set_ylabel("|accel| (g)")

axes[1].plot(shoulder_sig["time_s"], shoulder_sig["accel_mag_g"], color="tab:orange")
axes[1].set_title(f"{SHOULDER} — {shoulder_uid}")
axes[1].set_xlabel("time (s)")
axes[1].set_ylabel("|accel| (g)")

plt.tight_layout()
plt.show()

## 4. Per-recording summary stats

Peak accel magnitude, mean accel magnitude, duration, and rep count
(from `recordings.csv`) for every recording of each exercise — a quick
way to see how separable the two exercises are before building a
classifier.

In [ ]:
def summarize(record_uid: str) -> dict:
    sig = accel_magnitude(record_uid)
    return {
        "record_uid": record_uid,
        "duration_s": sig["time_s"].max() - sig["time_s"].min() if len(sig) else np.nan,
        "accel_mag_mean_g": sig["accel_mag_g"].mean(),
        "accel_mag_max_g": sig["accel_mag_g"].max(),
        "accel_mag_std_g": sig["accel_mag_g"].std(),
    }

def summary_table(recs: pd.DataFrame) -> pd.DataFrame:
    rows = [summarize(uid) for uid in recs["record_uid"]]
    out = pd.DataFrame(rows).merge(
        recs[["record_uid", "activity_reps", "master_sample_rows"]], on="record_uid"
    )
    return out

bench_summary = summary_table(bench_recordings)
shoulder_summary = summary_table(shoulder_recordings)

bench_summary.describe()

In [ ]:
shoulder_summary.describe()

## 5. Side-by-side comparison

Boxplots of peak accel magnitude per set, bench press vs shoulder press.

In [ ]:
compare = pd.concat([
    bench_summary.assign(exercise=BENCH),
    shoulder_summary.assign(exercise=SHOULDER),
])

compare.boxplot(column="accel_mag_max_g", by="exercise", figsize=(6, 5))
plt.title("Peak accel magnitude per set")
plt.suptitle("")
plt.ylabel("|accel| max (g)")
plt.show()

## Next steps

- Add the other exercise (`Lateral Raise`) to the comparison.
- Try gyroscope magnitude in addition to accelerometer.
- Segment individual reps within a set (using `activity_reps`) instead
  of only comparing whole-set summary stats.